In [1]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")

# Applying EDA
# df.head()

In [2]:
def revenue_filter(revenue):
    if pd.isna(revenue):
        return True
    
    revenue = str(revenue).lower()
    
    if ">500" in revenue:
        return False
    
    return True

df["revenue_valid"] = df["revenue"].apply(revenue_filter)

df_filtered = df[df["revenue_valid"]]

df_filtered[["company", "revenue"]]

,company,revenue
0,Vishwarupa Chemical Private Limited,<30
1,Vijay Chemical Industries (India) Private Limited,100-300
2,Galva Deco Parts Private Limited,200-300
3,Satyesh Brinechem Private Limited,200-300
4,Solenis Chemicals India Private Limited,300-500


In [3]:
def score_c1(description):
    if pd.isna(description):
        return ("Weak", 0)
    
    keywords = ["manufacture", "plant", "production", "facility"]
    
    if any(k in description.lower() for k in keywords):
        return ("Strong", 10)
    
    return ("Weak", 0)

In [4]:
def score_c2(city):
    if pd.isna(city):
        return ("Weak", 0)
    
    return ("Strong", 5)

In [5]:
def score_c3(description):
    if pd.isna(description):
        return ("Weak", 0)
    
    strong_keywords = ["r&d", "enzyme", "specialty", "proprietary", "fermentation"]
    
    if any(k in description.lower() for k in strong_keywords):
        return ("Strong", 25)
    
    return ("Moderate", 12.5)

In [6]:
def score_c4(founder):
    if pd.isna(founder) or founder.strip() == "":
        return ("Weak", 0)
    
    founder_lower = founder.lower()
    
    keywords_strong = ["phd", "dr.", "dr ", "engineer", "chemist", "scientist", "technologist"]
    
    if any(k in founder_lower for k in keywords_strong):
        return ("Strong", 20)
    
    return ("Moderate", 10)

In [7]:
def score_c5(segment):
    if pd.isna(segment):
        return ("Weak", 0)
    
    strong_segments = [
        "biotech", "specialty chemicals", "instrumentation",
        "medical devices", "performance chemicals"
    ]
    
    if any(s in segment.lower() for s in strong_segments):
        return ("Strong", 20)
    
    return ("Moderate", 10)

In [8]:
def score_c6(notes):
    if pd.isna(notes) or notes.strip() == "":
        return ("Weak", 0)
    
    notes = notes.lower()
    
    strong_keywords = [
        "rapid", "scale-up", "growth", "cagr", "expansion",
        "new plant", "capacity expansion", "funding", "hiring"
    ]
    
    moderate_keywords = [
        "export", "certification", "e-invoicing", "gst",
        "established", "stable"
    ]
    
    weak_keywords = [
        "decline", "negative", "loss", "drop", "decrease"
    ]
    
    # Strong signal → immediate strong
    if any(k in notes for k in strong_keywords):
        return ("Strong", 20)
    
    # Negative signal → weak
    if any(k in notes for k in weak_keywords):
        return ("Weak", 0)
    
    # Moderate signals
    if any(k in notes for k in moderate_keywords):
        return ("Moderate", 10)
    
    return ("Weak", 0)

In [9]:
results = []

for _, row in df.iterrows():
    
    c1, s1 = score_c1(row["description"])
    c2, s2 = score_c2(row["city"])
    c3, s3 = score_c3(row["description"])
    c4, s4 = score_c4(row["founder"])
    c5, s5 = score_c5(row["segment"])
    c6, s6 = score_c6(row["notes"])
    
    total = s1 + s2 + s3 + s4 + s5 + s6
    
    if total >= 80:
        verdict = "A - Strong Federer"
    elif total >= 60:
        verdict = "B - Probable"
    elif total >= 40:
        verdict = "C - Borderline"
    else:
        verdict = "D - Not ICP"
    
    results.append({
        "company": row["company"],
        "C1": c1,
        "C2": c2,
        "C3": c3,
        "C4": c4,
        "C5": c5,
        "C6": c6,
        "score": total,
        "verdict": verdict
    })

result_df = pd.DataFrame(results)

result_df

,company,C1,C2,C3,C4,C5,C6,score,verdict
0,Vishwarupa Chemical Private Limited,Strong,Strong,Moderate,Moderate,Strong,Weak,57.5,C - Borderline
1,Vijay Chemical Industries (India) Private Limited,Strong,Strong,Strong,Moderate,Strong,Strong,90.0,A - Strong Federer
2,Galva Deco Parts Private Limited,Strong,Strong,Moderate,Moderate,Moderate,Strong,67.5,B - Probable
3,Satyesh Brinechem Private Limited,Strong,Strong,Moderate,Moderate,Strong,Strong,77.5,B - Probable
4,Solenis Chemicals India Private Limited,Strong,Strong,Strong,Moderate,Strong,Strong,90.0,A - Strong Federer


In [10]:
result_df.to_csv("scored_companies.csv", index=False)

print("Scored file saved as scored_companies.csv")

Scored file saved as scored_companies.csv
